# 11 · Window Classifier Separability

This notebook turns the descriptor-space geometry from Notebook 10 into a simple classification benchmark.

We ask three binary questions:

1. **zeta vs Poisson**
2. **GUE vs Poisson**
3. **zeta vs GUE**

If local descriptor vectors can easily separate zeta from Poisson but struggle to separate zeta from GUE, that supports the idea that zeta and GUE share a common local statistical structure.

## What this notebook does

- rebuild 5-gap descriptor vectors
- standardize the feature space
- train simple classifiers
- evaluate on held-out test splits
- summarize which pairings are easy or hard to distinguish

This is an exploratory lab notebook, not a benchmark suite.
The point is to get an interpretable separability signal, not to optimize models aggressively.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Data sources

In [ ]:
# Zeta gaps
N = 700
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

# Poisson control
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))

# Finite GUE bulk gaps
def sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep

        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=rng)

len(zeta_gaps), len(poisson_gaps), len(gue_gaps)

## Build 5-gap windows and descriptor vectors

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_features(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r)), float(np.std(r))

def descriptor_vector(window):
    ratio_mean, ratio_std = ratio_features(window)
    return np.array([
        window[0], window[1], window[2], window[3], window[4],
        unevenness(window),
        reversal_symmetry_score(window),
        window_entropy(window),
        ratio_mean,
        ratio_std,
    ], dtype=float)

def build_descriptors(gaps, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    desc = np.array([descriptor_vector(w) for w in windows_n])
    return windows_n, desc

zeta_w, zeta_desc = build_descriptors(zeta_gaps, k=5)
poisson_w, poisson_desc = build_descriptors(poisson_gaps, k=5)
gue_w, gue_desc = build_descriptors(gue_gaps, k=5)

zeta_desc.shape, poisson_desc.shape, gue_desc.shape

## Utility functions

We implement:

- a train/test split
- feature standardization based on the training set
- a very simple nearest-centroid classifier
- a basic k-nearest-neighbors classifier (k=5)

The goal is interpretability and robustness, not sophistication.

In [ ]:
def train_test_split_binary(X0, X1, test_frac=0.3, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    n0 = len(X0)
    n1 = len(X1)

    idx0 = rng.permutation(n0)
    idx1 = rng.permutation(n1)

    t0 = int((1 - test_frac) * n0)
    t1 = int((1 - test_frac) * n1)

    train0, test0 = X0[idx0[:t0]], X0[idx0[t0:]]
    train1, test1 = X1[idx1[:t1]], X1[idx1[t1:]]

    X_train = np.vstack([train0, train1])
    y_train = np.array([0] * len(train0) + [1] * len(train1))

    X_test = np.vstack([test0, test1])
    y_test = np.array([0] * len(test0) + [1] * len(test1))

    return X_train, y_train, X_test, y_test

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

def nearest_centroid_fit(X_train, y_train):
    classes = np.unique(y_train)
    centroids = {c: X_train[y_train == c].mean(axis=0) for c in classes}
    return centroids

def nearest_centroid_predict(X, centroids):
    classes = sorted(centroids.keys())
    centroid_array = np.vstack([centroids[c] for c in classes])
    dists = np.linalg.norm(X[:, None, :] - centroid_array[None, :, :], axis=2)
    pred_idx = np.argmin(dists, axis=1)
    return np.array([classes[i] for i in pred_idx])

def knn_predict(X_train, y_train, X_test, k=5):
    preds = []
    for x in X_test:
        d = np.linalg.norm(X_train - x, axis=1)
        nn = np.argsort(d)[:k]
        votes = y_train[nn]
        counts = np.bincount(votes, minlength=2)
        preds.append(np.argmax(counts))
    return np.array(preds)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def confusion_2x2(y_true, y_pred):
    tp00 = np.sum((y_true == 0) & (y_pred == 0))
    tp01 = np.sum((y_true == 0) & (y_pred == 1))
    tp10 = np.sum((y_true == 1) & (y_pred == 0))
    tp11 = np.sum((y_true == 1) & (y_pred == 1))
    return np.array([[tp00, tp01], [tp10, tp11]])

## Run the three binary tasks

We evaluate:

- zeta vs Poisson
- GUE vs Poisson
- zeta vs GUE

using both:
- nearest-centroid
- k-nearest-neighbors (k=5)

In [ ]:
def run_binary_task(name, X0, X1, rng):
    X_train, y_train, X_test, y_test = train_test_split_binary(X0, X1, test_frac=0.3, rng=rng)
    X_train_s, X_test_s = standardize_train_test(X_train, X_test)

    centroids = nearest_centroid_fit(X_train_s, y_train)
    y_pred_centroid = nearest_centroid_predict(X_test_s, centroids)

    y_pred_knn = knn_predict(X_train_s, y_train, X_test_s, k=5)

    return {
        "task": name,
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "centroid_accuracy": accuracy(y_test, y_pred_centroid),
        "knn_accuracy": accuracy(y_test, y_pred_knn),
        "centroid_confusion": confusion_2x2(y_test, y_pred_centroid),
        "knn_confusion": confusion_2x2(y_test, y_pred_knn),
    }

results = [
    run_binary_task("zeta vs Poisson", zeta_desc, poisson_desc, rng),
    run_binary_task("GUE vs Poisson", gue_desc, poisson_desc, rng),
    run_binary_task("zeta vs GUE", zeta_desc, gue_desc, rng),
]

results

## Accuracy summary

In [ ]:
task_names = [r["task"] for r in results]
centroid_acc = [r["centroid_accuracy"] for r in results]
knn_acc = [r["knn_accuracy"] for r in results]

x = np.arange(len(task_names))
width = 0.35

plt.figure(figsize=(8.6, 5.0))
plt.bar(x - width/2, centroid_acc, width, label="nearest centroid")
plt.bar(x + width/2, knn_acc, width, label="k-NN (k=5)")
plt.xticks(x, task_names, rotation=15)
plt.ylim(0.4, 1.0)
plt.ylabel("accuracy")
plt.title("Binary window-classification accuracy")
plt.legend()
plt.show()

## Confusion matrices

In [ ]:
for r in results:
    print("\n" + "="*60)
    print(r["task"])
    print("Nearest-centroid confusion:")
    print(r["centroid_confusion"])
    print("k-NN confusion:")
    print(r["knn_confusion"])

## Repeat over several random splits

One train/test split can be noisy.  
Here we average over multiple random splits to get a more stable picture.

In [ ]:
def repeat_task(X0, X1, n_repeats=12):
    centroid_scores = []
    knn_scores = []
    local_rng = np.random.default_rng(9423)

    for _ in range(n_repeats):
        X_train, y_train, X_test, y_test = train_test_split_binary(X0, X1, test_frac=0.3, rng=local_rng)
        X_train_s, X_test_s = standardize_train_test(X_train, X_test)

        centroids = nearest_centroid_fit(X_train_s, y_train)
        y_pred_centroid = nearest_centroid_predict(X_test_s, centroids)
        y_pred_knn = knn_predict(X_train_s, y_train, X_test_s, k=5)

        centroid_scores.append(accuracy(y_test, y_pred_centroid))
        knn_scores.append(accuracy(y_test, y_pred_knn))

    return np.array(centroid_scores), np.array(knn_scores)

repeat_summary = {}

for name, X0, X1 in [
    ("zeta vs Poisson", zeta_desc, poisson_desc),
    ("GUE vs Poisson", gue_desc, poisson_desc),
    ("zeta vs GUE", zeta_desc, gue_desc),
]:
    c_scores, k_scores = repeat_task(X0, X1, n_repeats=12)
    repeat_summary[name] = {
        "centroid_mean": float(c_scores.mean()),
        "centroid_std": float(c_scores.std()),
        "knn_mean": float(k_scores.mean()),
        "knn_std": float(k_scores.std()),
    }

repeat_summary

## Mean accuracy with error bars

In [ ]:
tasks = list(repeat_summary.keys())
centroid_means = [repeat_summary[t]["centroid_mean"] for t in tasks]
centroid_stds = [repeat_summary[t]["centroid_std"] for t in tasks]
knn_means = [repeat_summary[t]["knn_mean"] for t in tasks]
knn_stds = [repeat_summary[t]["knn_std"] for t in tasks]

x = np.arange(len(tasks))
width = 0.35

plt.figure(figsize=(8.8, 5.2))
plt.bar(x - width/2, centroid_means, width, yerr=centroid_stds, capsize=4, label="nearest centroid")
plt.bar(x + width/2, knn_means, width, yerr=knn_stds, capsize=4, label="k-NN (k=5)")
plt.xticks(x, tasks, rotation=15)
plt.ylim(0.4, 1.0)
plt.ylabel("mean accuracy")
plt.title("Repeated-split classification accuracy")
plt.legend()
plt.show()

## PCA visualization for one task: zeta vs GUE

We use a quick PCA projection just for visual context.

In [ ]:
def pca_2d(X):
    Xc = X - X.mean(axis=0)
    cov = np.cov(Xc, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvecs = eigvecs[:, order]
    comps = eigvecs[:, :2]
    return Xc @ comps

X_zg = np.vstack([zeta_desc, gue_desc])
y_zg = np.array([0] * len(zeta_desc) + [1] * len(gue_desc))

X_zg_std = (X_zg - X_zg.mean(axis=0)) / X_zg.std(axis=0)
X_zg_pca = pca_2d(X_zg_std)

plt.figure(figsize=(8.6, 5.8))
plt.scatter(X_zg_pca[y_zg == 0, 0], X_zg_pca[y_zg == 0, 1], s=10, alpha=0.30, label="zeta")
plt.scatter(X_zg_pca[y_zg == 1, 0], X_zg_pca[y_zg == 1, 1], s=10, alpha=0.30, label="GUE")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Descriptor PCA: zeta vs GUE")
plt.legend()
plt.show()

## Summary table

In [ ]:
summary_table = {
    task: {
        "centroid_mean_accuracy": repeat_summary[task]["centroid_mean"],
        "centroid_std_accuracy": repeat_summary[task]["centroid_std"],
        "knn_mean_accuracy": repeat_summary[task]["knn_mean"],
        "knn_std_accuracy": repeat_summary[task]["knn_std"],
    }
    for task in repeat_summary
}
summary_table

## Notes

- If zeta vs Poisson and GUE vs Poisson are easy, but zeta vs GUE is much harder, that supports the idea that the descriptor space is capturing random-matrix-style structure rather than arbitrary artifacts.
- The classifiers here are intentionally simple; strong separation with simple models is often more convincing than overfitting with complex ones.
- A low zeta-vs-GUE accuracy does not prove identity, but it does support the claim that local window descriptors fail to distinguish them easily.

## Next directions

A natural next notebook could do one of these:

1. compute geometric distances and neighborhood overlap in embedding space  
2. bootstrap significance for centroid differences  
3. test alternative descriptor sets  
4. add conditional-window classification (after small vs after large gap)